# NB-26 — the control the negation result is missing

**The review's point 7, and it is fair.** The paper compares deleting the
negation particle (T1) against retargeting the whole sentence (T5) and reports
that T1 costs at least 9.3x less perturbation. But a one-word edit will almost
always be cheaper than a full rewrite, so most of that ratio may be measuring
*edit distance*, not anything about negation. Without a control the section has
no Bengali-specific finding in it.

**The control.** On the **same 38 utterances**, delete a different single word —
one that is not a negator, matched as closely as possible to the particle on
length and position. Call it **T6**. Then:

| comparison | what it isolates |
|---|---|
| T1 vs T5 (already measured) | one-word edit against a full rewrite — mostly edit distance |
| **T1 vs T6** (this notebook) | **one-word edit against one-word edit — negation itself** |

**The two outcomes, and both are reportable.**

- **T1 is still clearly cheaper than T6.** Negation is special: the particle
  carries meaning far out of proportion to its acoustic size, and you have a
  real low-resource-language finding.
- **T1 and T6 cost about the same.** The 9.34x is an edit-distance result.
  Say so, retitle the section, and the security point survives untouched —
  an attacker can still flip a sentence's meaning for a ninth of the cost of
  rewriting it. That claim never depended on negation being unique.

**Cost.** T1 and T5 are already measured on both checkpoints; only T6 is new.
Two runs over 38 utterances, so budget 3–4 GPU-hours.

**What the control word selection does, and its limits.** It tokenises with the
sentence-final *danda* detached (a plain whitespace split misses every
sentence-final particle — the defect that took NB-03's candidate pool from 71 to
277), excludes every Bengali negator it knows about (না, নি, নয়, নেই, নাই), and
then picks the remaining word closest in character length and position to the
particle. It is deterministic. It is also mechanical: it does not know whether
deleting that particular word leaves a grammatical sentence. That is a stated
limitation, not a hidden one — and it cuts *against* the negation hypothesis,
since an ungrammatical control is if anything easier to reach, not harder.


## 1 · Configuration

In [ ]:
NOTEBOOK = "NB-26-negation-control"

# T6 only. T1 and T5 are already measured on both of these checkpoints.
EDIT_TYPE = "T6"
# `t1_csv` is the EXACT published filename, not something derived from `tag`.
# The two T1 runs were written by different notebooks with different naming:
#   NB-05 (100 h): eps50_rung100_final_T1.csv        -- no _n suffix
#   NB-13 (1 h)  : eps50_rung1_final_T1_n38.csv      -- with _n38
# Both use an UNDERSCORE between rung and checkpoint; `tag` uses a hyphen. A
# filename derived from `tag` matches neither, and the paired comparison would
# silently report "T1 not attached" after the GPU had already been spent.
QUEUE = [
    dict(tag="rung100-final", ckpt_dataset="ckpt-xlsr-100h", rung=100,
         checkpoint="final", t1_ref="100% (38/38)", t5_ref="36.8% (14/38)",
         t1_csv="eps50_rung100_final_T1.csv",
         t1_dataset="eps50-rung100-final-t1-paired38"),
    dict(tag="rung1-final",   ckpt_dataset="ckpt-xlsr-1h",   rung=1,
         checkpoint="final", t1_ref="84.2% (32/38)", t5_ref="63.2% (24/38)",
         t1_csv="eps50_rung1_final_T1_n38.csv",
         t1_dataset="eps50-rung1-t1t5-nb03"),
]

BATCH_SIZE     = 8
USE_AMP        = True
FIXED_EPS      = None
PROBE_STEPS    = 400
CONFIRM_STEPS  = 800
STALL_PATIENCE = 100
CHECK_EVERY    = 50
LR             = 1e-4
SEED           = 1337
WALL_HOURS     = 8.0
VALIDATE_BATCHING = False

# T6 is a one-word deletion, exactly like T1, so it needs T1's bracket. Using
# T5's floor of 1e-4 would pin most values to it and turn the comparison into
# two upper bounds -- which is the bug NB-13's comment records for T1.
EPS_LO, EPS_HI = 1e-5, 2e-3
BISECT_ROUNDS  = 6

assert EDIT_TYPE == "T6"
assert (EPS_LO, BISECT_ROUNDS) == (1e-5, 6), (
    "T6 must use T1's bracket and rounds. Anything else makes the T1-vs-T6 "
    "ratio incomparable to the published 9.34x, which is the whole point.")

print(f"{NOTEBOOK} | {EDIT_TYPE} on the 38-utterance NB-03 spine | "
      f"{len(QUEUE)} checkpoints")
print("  already measured on these checkpoints:")
for q in QUEUE:
    print(f"    {q['tag']:16s} T1 {q['t1_ref']:16s} T5 {q['t5_ref']}")
print("  published paired ratio T1:T5 = 9.34x at 100 h, 4.33x at 1 h")

## 2 · Boilerplate

In [ ]:
import os, hashlib
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # attacks optimise the INPUT: never DataParallel
import sys, json, time, random, re, glob, unicodedata, gc
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP, DEVICE = "/kaggle/working", "/kaggle/temp", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"

# PROV holds only what is defined AT THIS POINT. In NB-05 the rung, edit type
# and bracket were module-level constants, so they could be read here; in a
# queue runner they are per-condition and are not bound yet. Reading them here
# is a NameError that fires three cells before anything expensive starts. The
# run-specific fields are added by the summary cell at the end.
PROV = {"notebook": NOTEBOOK, "seed": SEED,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# See NB-01 for the full explanation: pinning an old `accelerate`/`datasets`
# silently downgrades numpy mid-session and breaks pandas much later with
# "No module named 'numpy.rec'". Pin numpy to what is already loaded.
# `datasets` is not imported by this notebook, so it is not installed.
import numpy, subprocess, sys
NP_BEFORE = numpy.__version__

!pip install -q transformers==4.44.2 librosa soundfile jiwer "numpy=={NP_BEFORE}" 2>&1 | tail -2

NP_DISK = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
assert NP_BEFORE == NP_DISK, f"numpy moved {NP_BEFORE} -> {NP_DISK}; relax the pin that forced it."
print(f"numpy {NP_BEFORE} stable")

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa
print("transformers", transformers.__version__)
PROV["transformers"] = transformers.__version__
PROV["numpy"] = NP_BEFORE


## 3 · Inputs, the 38-utterance spine, and building the T6 targets

Stage A is pure CPU and deterministic: no model is loaded yet. It reads NB-03's
hand-validated corpus, finds the negation particle in each source sentence, and
writes the matched control deletion beside it.

Every rejected utterance is reported with its reason. A silent drop here would
quietly unpair the comparison.

In [ ]:
inputs = sorted({p for pat in ("/kaggle/input/*", "/kaggle/input/*/*",
                               "/kaggle/input/*/*/*")
                 for p in glob.glob(pat) if os.path.isdir(p)})

SPLITS = next((p for p in inputs if "splits" in p.lower()), None)
assert SPLITS, "attach bn-asr-splits"
EV_CSV = next((f"{p}/attack_eval_t1.csv" for p in inputs
               if os.path.exists(f"{p}/attack_eval_t1.csv")), None)
assert EV_CSV, ("attack_eval_t1.csv not found. Attach bn-attack-targets (NB-03's "
                f"output). Visible: {[os.path.basename(p) for p in inputs]}")
TGT_CSV = next((f"{p}/bn_attack_targets.csv" for p in inputs
                if os.path.exists(f"{p}/bn_attack_targets.csv")), None)
assert TGT_CSV, "bn_attack_targets.csv not found -- attach bn-attack-targets"
print("eval set:", EV_CSV)
print("targets :", TGT_CSV)

ZW = dict.fromkeys(map(ord, "‌‍"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

def resolve_audio(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0): return df
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    c = [d for pat in (f"/kaggle/input/*/{stem}", f"/kaggle/input/*/*/{stem}",
                       f"/kaggle/input/*/*/*/{stem}")
         for d in glob.glob(pat) if os.path.isdir(d)]
    assert c, f"audio dir '{stem}' not found -- attach bengaliai-speech"
    df = df.copy(); df["path"] = c[0] + "/" + df["utt_id"].astype(str) + ext
    return df

base = resolve_audio(pd.read_csv(EV_CSV))
tg_all = pd.read_csv(TGT_CSV)

# The T1 rows define the spine: these are the utterances the published ratio
# was measured on. T6 must run on exactly these, or the comparison is unpaired.
t1 = tg_all[tg_all.edit_type == "T1"][["source_utterance_id", "target_text"]]
spine = base.merge(t1, left_on="utt_id", right_on="source_utterance_id", how="inner")
spine["t1_text"] = spine.target_text.map(norm_bn)
spine = spine.drop(columns=["target_text"])
print(f"\nNB-03 T1 spine: {len(spine)} utterances")
assert len(spine) == 38, (
    f"expected the published 38-utterance spine, got {len(spine)}. The ratio this "
    "notebook produces is only comparable to 9.34x on the same corpus.")

### The control-word chooser

In [ ]:
DANDA = "\u0964"                      # ।  the sentence-final danda
NEG   = norm_bn("\u09a8\u09be")       # না  the post-verbal negation particle

# Every other Bengali negator we know of. The control word must never be one of
# these, or "delete a non-negation word" quietly deletes a different negation.
# Normalised through norm_bn so the comparison is NFC-to-NFC: নয় in particular
# has two encodings and only one of them would match a raw literal.
OTHER_NEG = {norm_bn(s) for s in
             ["\u09a8\u09bf",               # নি
              "\u09a8\u09df",               # নয়  (single codepoint ya+nukta)
              "\u09a8\u09af\u09bc",         # নয়  (ya + combining nukta)
              "\u09a8\u09c7\u0987",         # নেই
              "\u09a8\u09be\u0987"]}        # নাই


def tokenise(sent):
    """Whitespace split, with the sentence-final danda detached.

    A plain split leaves 'না।' as a single token, which is why NB-03's first
    pass found 71 candidates where there were 277 -- it missed every
    sentence-final particle, and sentence-final is the commonest position.
    Returns (tokens, trailing_danda).
    """
    s = norm_bn(sent)
    tail = ""
    if s.endswith(DANDA):
        s, tail = s[:-1].rstrip(), DANDA
    return s.split(), tail


def detokenise(toks, tail):
    return (" ".join(toks) + tail).strip()


def delete_token(sent, idx):
    toks, tail = tokenise(sent)
    assert 0 <= idx < len(toks), f"index {idx} outside {len(toks)} tokens"
    return detokenise(toks[:idx] + toks[idx + 1:], tail)


def find_negation(sent):
    """Index of the standalone negation particle, or None.

    Takes the LAST occurrence: the particle is post-verbal, and a sentence with
    two of them is negating the later clause.
    """
    toks, _ = tokenise(sent)
    hits = [i for i, t in enumerate(toks) if t == NEG]
    return hits[-1] if hits else None


def pick_control(sent, neg_idx):
    """Pick a NON-negation word to delete, matched to the particle.

    Ranked by, in order:
      1. not a negator of any kind
      2. character length closest to the particle's
      3. position closest to the particle's
      4. lowest index, so the choice is deterministic across runs

    Returns (index, token, why) or (None, None, why not).

    Limitation, stated rather than hidden: this does not check that the result
    is grammatical. An ungrammatical control is if anything EASIER to reach than
    a grammatical one, so the bias runs against the negation hypothesis, not
    for it.
    """
    toks, _ = tokenise(sent)
    if len(toks) < 3:
        return None, None, "too short: deleting a word would leave under 2 tokens"
    target_len = len(toks[neg_idx])
    cands = [(i, t) for i, t in enumerate(toks)
             if i != neg_idx and t != NEG and t not in OTHER_NEG]
    if not cands:
        return None, None, "no non-negation token available"
    i, t = min(cands, key=lambda it: (abs(len(it[1]) - target_len),
                                      abs(it[0] - neg_idx), it[0]))
    return i, t, f"len {len(t)} vs {target_len}, offset {i - neg_idx:+d} from the particle"


# Self-test on constructed sentences, so a broken chooser fails here and not
# after the GPU has been paid for.
_t = "\u0986\u09ae\u09bf \u09ad\u09be\u09a4 \u0996\u09be\u0987 \u09a8\u09be\u0964"
_ni = find_negation(_t)
assert _ni == 3, f"negation index {_ni}, expected 3"
assert delete_token(_t, _ni).endswith(DANDA), "danda lost on deletion"
_ci, _ct, _ = pick_control(_t, _ni)
assert _ct not in OTHER_NEG and _ct != NEG, "chooser returned a negator"
_t2 = "\u09a4\u09bf\u09a8\u09bf \u09ac\u09b2\u09c7\u09a8 \u09a8\u09bf \u09a8\u09be\u0964"
_ci2, _ct2, _ = pick_control(_t2, find_negation(_t2))
assert _ct2 not in OTHER_NEG, f"chooser picked the negator {_ct2!r}"
assert find_negation("\u0986\u09ae\u09bf \u09ad\u09be\u09a4 \u0996\u09be\u0987") is None
print("control-word chooser self-test passed")


### Build and audit the T6 target set

In [ ]:
rows, rejected = [], []
for _, r in spine.iterrows():
    src = norm_bn(r["text"])
    ni = find_negation(src)
    if ni is None:
        rejected.append((r["utt_id"], "no standalone negation particle")); continue
    ci, ct, why = pick_control(src, ni)
    if ci is None:
        rejected.append((r["utt_id"], why)); continue
    t6 = delete_token(src, ci)
    if norm_bn(t6) == src or not t6.strip():
        rejected.append((r["utt_id"], "control deletion was a no-op")); continue
    toks, _ = tokenise(src)
    rows.append(dict(utt_id=r["utt_id"], path=r["path"], duration=r["duration"],
                     text=src, target_text=t6,
                     neg_idx=ni, neg_token=toks[ni],
                     ctl_idx=ci, ctl_token=ct, match=why,
                     source_tokens=len(toks), target_tokens=len(toks) - 1))

ev = pd.DataFrame(rows).reset_index(drop=True)
print(f"T6 targets built for {len(ev)}/{len(spine)} utterances")
if rejected:
    print(f"\n{len(rejected)} rejected:")
    for u, why in rejected:
        print(f"   {u}  {why}")

assert len(ev) >= 30, (
    f"only {len(ev)} T6 targets. Below about 30 the paired test loses the power "
    "to say anything, and the control stops being worth the GPU time.")

# Audit: the control must never be a negator, and must differ from T1.
for _, r in ev.iterrows():
    assert r.ctl_token != NEG and r.ctl_token not in OTHER_NEG, \
        f"{r.utt_id}: control token {r.ctl_token!r} is a negator"
    assert r.ctl_idx != r.neg_idx, f"{r.utt_id}: control is the particle itself"
print("\naudit passed: no control word is a negator, none is the particle")

print("\n  deleted-token length, particle vs control:")
print(f"    particle: mean {ev.neg_token.str.len().mean():.2f} chars")
print(f"    control : mean {ev.ctl_token.str.len().mean():.2f} chars")
print(f"  position offset (control - particle): "
      f"median {int((ev.ctl_idx - ev.neg_idx).median()):+d} tokens")
print("\n  first five:")
for _, r in ev.head().iterrows():
    print(f"    {r.utt_id}")
    print(f"      source : {r.text}")
    print(f"      T6     : {r.target_text}        [dropped {r.ctl_token!r}; {r.match}]")
ev.to_csv(f"{WORK}/t6_targets_n{len(ev)}.csv", index=False)

## 4 · Front end, loss, PGD, bisection — unchanged from NB-13

In [ ]:
SR = 16000

def load_batch(rows):
    """Length-bucketed batch. Returns padded waveforms + a validity mask."""
    wavs = []
    for _, r in rows.iterrows():
        a, _ = librosa.load(r["path"], sr=SR, mono=True)
        a = np.asarray(a, np.float32)
        pk = np.abs(a).max()
        wavs.append(a / max(pk, 1.0))
    T = max(len(w) for w in wavs)
    x = torch.zeros(len(wavs), T, dtype=torch.float32)
    m = torch.zeros(len(wavs), T, dtype=torch.float32)
    for i, w in enumerate(wavs):
        x[i, :len(w)] = torch.from_numpy(w); m[i, :len(w)] = 1.0
    return x.to(DEVICE), m.to(DEVICE)


def frontend_b(x, mask):
    """Masked zero-mean/unit-variance. Computing stats over the PADDING would
    corrupt short utterances in a mixed-length batch."""
    if DO_NORMALIZE:
        n = mask.sum(-1, keepdim=True).clamp(min=1)
        mu = (x * mask).sum(-1, keepdim=True) / n
        var = (((x - mu) * mask) ** 2).sum(-1, keepdim=True) / n
        x = (x - mu) / torch.sqrt(var + 1e-7)
        x = x * mask
    return x


def logits_b(x, mask):
    return model(input_values=frontend_b(x, mask),
                 attention_mask=mask.long()).logits


def ctc_per_sample(x, mask, tgt_pad, tgt_len):
    """Per-sample CTC loss -> lets us mask out finished samples."""
    lg = logits_b(x, mask)
    logp = F.log_softmax(lg.float(), -1).transpose(0, 1)          # (L,B,V)
    in_len = model._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
    in_len = in_len.clamp(max=logp.shape[0])
    return F.ctc_loss(logp, tgt_pad, in_len, tgt_len,
                      blank=BLANK_ID, zero_infinity=True, reduction="none")


def decode_b(x, mask):
    with torch.no_grad():
        ids = torch.argmax(logits_b(x, mask), -1)
    return [norm_bn(s) for s in processor.batch_decode(ids)]


def int16_rt(x):
    return torch.clamp((x * 32767.0).round(), -32768.0, 32767.0) / 32767.0

def snr_db_b(clean, adv, mask):
    n = (adv - clean) * mask
    s = clean * mask
    return (10 * torch.log10(s.pow(2).sum(-1) / (n.pow(2).sum(-1) + 1e-12))).cpu().numpy()

In [ ]:
def pgd_batch(x, mask, tgt_pad, tgt_len, targets, eps_vec,
              steps, delta_init=None, lr=LR):
    """eps_vec: (B,) per-sample L-inf budgets.
       Returns (success bool array, frozen deltas, steps_used)."""
    B = x.shape[0]
    eps_col = eps_vec.view(-1, 1)
    delta = (torch.zeros_like(x) if delta_init is None
             else torch.max(torch.min(delta_init.clone(), eps_col), -eps_col))
    delta.requires_grad_(True)
    opt = torch.optim.Adam([delta], lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    active = torch.ones(B, dtype=torch.bool, device=DEVICE)
    frozen = torch.zeros_like(x)
    success = np.zeros(B, dtype=bool)
    best_loss = torch.full((B,), float("inf"), device=DEVICE)
    stall = torch.zeros(B, dtype=torch.long, device=DEVICE)

    for step in range(steps):
        if not active.any():
            break
        x_adv = torch.clamp(x + delta, -1, 1) * mask
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            per = ctc_per_sample(x_adv, mask, tgt_pad, tgt_len)
        loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)

        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt)
        if delta.grad is not None:
            delta.grad[~active] = 0          # finished samples stop moving
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            delta.data = torch.max(torch.min(delta.data, eps_col), -eps_col)

        with torch.no_grad():                # stall detection
            improved = per < best_loss - 1e-3
            best_loss = torch.minimum(best_loss, per.detach())
            stall = torch.where(improved, torch.zeros_like(stall), stall + 1)
            give_up = active & (stall > STALL_PATIENCE)
            active = active & ~give_up

        if (step + 1) % CHECK_EVERY == 0:
            hyp = decode_b(int16_rt(torch.clamp(x + delta, -1, 1)) * mask, mask)
            with torch.no_grad():
                for i in range(B):
                    if active[i] and hyp[i] == targets[i]:
                        frozen[i] = delta[i].detach()
                        success[i] = True
                        active[i] = False
    return success, frozen, step + 1

In [ ]:
def eps50_batch(x, mask, tgt_pad, tgt_len, targets, verbose=True):
    """Per-sample geometric bisection for the smallest budget that still works.

    ---------------------------------------------------------------------------
    BUG FIXED 2026-09-22 (found in the rung-1 T5 run, 40% censoring).

    The previous version ended with

        ok, best, _ = pgd_batch(..., hi, CONFIRM_STEPS, delta_init=best)

    which REASSIGNED `best` wholesale to pgd_batch's return value. That return
    is a `frozen` buffer holding zeros for any sample that did not succeed on
    that particular pass -- so one unlucky final pass wiped out a delta that had
    already been verified at a larger budget. Those samples were then written
    out as censored, with SNR 70-86 dB (the signature of delta == 0) even though
    the ceiling probe had already proved them attackable.

    In the rung-1 run that discarded ~6 of 25 real measurements. Worse, it does
    so at a RATE THAT VARIES BY MODEL, so comparing median eps50 across rungs
    would have been comparing differently-biased subsamples -- straight into
    RQ1's headline.

    Fix: keep a running `best_eps` / `best_delta` that is only ever updated on a
    VERIFIED success. Nothing overwrites a confirmed result. Censoring now means
    exactly one thing: the attack failed at every budget tried, ceiling included.

    Side benefit: eps50 becomes a guaranteed UPPER BOUND on the true minimum.
    Optimiser noise can only make it conservative, never wrong in an
    unpredictable direction -- which is the property you want in a metric you
    are about to compare across four models.
    ---------------------------------------------------------------------------
    """
    B = x.shape[0]
    lo = torch.full((B,), EPS_LO, device=DEVICE)
    hi = torch.full((B,), EPS_HI, device=DEVICE)
    best_eps   = torch.full((B,), float("nan"), device=DEVICE)   # verified only
    best_delta = torch.zeros_like(x)

    def record(okt, eps_vec, d):
        """Adopt a result only when it is both successful and better."""
        nonlocal best_eps, best_delta
        better = okt & (torch.isnan(best_eps) | (eps_vec < best_eps))
        best_eps   = torch.where(better, eps_vec, best_eps)
        best_delta = torch.where(better.view(-1, 1), d, best_delta)

    # Ceiling probe. Failure here is genuine right-censoring.
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, hi, CONFIRM_STEPS)
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, hi, d)
    feasible = ok.copy()
    if verbose:
        print(f"  ceiling probe eps={EPS_HI:.1e}: {feasible.sum()}/{B} feasible, "
              f"{(~feasible).sum()} censored")

    for rnd in range(BISECT_ROUNDS):
        mid = torch.sqrt(lo * hi)
        ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                             PROBE_STEPS, delta_init=best_delta)
        ok = ok & feasible
        okt = torch.tensor(ok, device=DEVICE)
        hi = torch.where(okt, mid, hi)
        lo = torch.where(okt, lo, mid)
        record(okt, mid, d)
        if verbose:
            print(f"  round {rnd+1}/{BISECT_ROUNDS}: {ok.sum()}/{B} succeeded "
                  f"| median eps {mid.median().item():.3e}")

    # Budget-consistency round: one more bisection at the FULL step budget, so
    # the reported boundary is measured at the budget eps50 is defined at rather
    # than at the cheaper probe budget. See RUNLOG for why this matters.
    #
    # Skipped in FIXED_EPS mode: there is no bracket to refine. The ceiling probe
    # already ran at CONFIRM_STEPS, so the single reported budget is measured at
    # the full budget and needs no second pass.
    if BISECT_ROUNDS == 0:
        eps       = best_eps.cpu().numpy()
        confirmed = ~np.isnan(eps)
        if verbose:
            print(f"  -> FIXED_EPS {EPS_HI:.1e}: {confirmed.sum()}/{B} succeeded, "
                  f"{(~confirmed).sum()} resisted")
        return eps, np.ones(B), confirmed, best_delta

    mid = torch.sqrt(lo * hi)
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                         CONFIRM_STEPS, delta_init=best_delta)
    ok = ok & feasible
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, mid, d)
    if verbose:
        print(f"  budget-consistency round @ {CONFIRM_STEPS} steps: {ok.sum()}/{B}")

    eps       = best_eps.cpu().numpy()
    confirmed = ~np.isnan(eps)                 # censored == never succeeded
    width     = (hi / lo).cpu().numpy()
    if verbose:
        print(f"  -> {confirmed.sum()}/{B} measured, {(~confirmed).sum()} censored")
    return eps, width, confirmed, best_delta


## 5 · Run T6 on both checkpoints

In [ ]:
def load_ckpt(cfg):
    global model, processor, BLANK_ID, DO_NORMALIZE, MODEL_DIR, RUNG, CHECKPOINT
    cands = [p for p in inputs
             if re.search(rf"{re.escape(cfg['ckpt_dataset'])}$", os.path.basename(p))]
    assert len(cands) == 1, (f"need exactly one input matching "
                             f"'{cfg['ckpt_dataset']}', got "
                             f"{[os.path.basename(x) for x in cands]}")
    ck = cands[0]
    MODEL_DIR = ck if os.path.exists(f"{ck}/config.json") else f"{ck}/final"
    assert os.path.exists(f"{MODEL_DIR}/config.json"), f"no config.json under {MODEL_DIR}"
    processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
    model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad_(False)
    model.config.apply_spec_augment = False
    DO_NORMALIZE = bool(getattr(processor.feature_extractor, "do_normalize", True))
    BLANK_ID = model.config.pad_token_id
    RUNG, CHECKPOINT = cfg["rung"], cfg["checkpoint"]
    print(f"  loaded {MODEL_DIR} | blank {BLANK_ID} | do_normalize {DO_NORMALIZE}")


SESSION_START = time.time()
DEADLINE = SESSION_START + WALL_HOURS * 3600
summary = []

for qi, cfg in enumerate(QUEUE, 1):
    print("\n" + "=" * 74)
    print(f"[{qi}/{len(QUEUE)}] {cfg['tag']}  {EDIT_TYPE}   "
          f"({(DEADLINE-time.time())/3600:.1f} h left)")
    print("=" * 74)
    # Same shape NB-13 used, so T1 and T6 sit side by side in one convention.
    OUT = (f"{WORK}/eps50_rung{cfg['rung']}_{cfg['checkpoint']}"
           f"_{EDIT_TYPE}_n{len(ev)}.csv")

    results, seen = [], set()
    for d in range(1, 5):
        for p in sorted(glob.glob("/kaggle/input/" + "*/" * d + os.path.basename(OUT))):
            for r in pd.read_csv(p).to_dict("records"):
                if r["utt_id"] not in seen:
                    seen.add(r["utt_id"]); results.append(r)
            print(f"  resume source: {p}")
    if results:
        print(f"  resuming: {len(results)} already measured")

    load_ckpt(cfg)
    todo = ev[~ev.utt_id.isin(seen)].reset_index(drop=True)
    batches = [todo.iloc[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
    print(f"  {len(todo)} remaining -> {len(batches)} batches")

    for bi, sub in enumerate(batches):
        if time.time() > DEADLINE:
            print("  *** wall clock reached ***"); break
        xb, mb = load_batch(sub)
        tp = [torch.tensor(processor.tokenizer(t).input_ids, device=DEVICE)
              for t in sub.target_text]
        tl = torch.tensor([len(t) for t in tp], device=DEVICE)
        tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
        t0 = time.time()
        eps, width, conf, best = eps50_batch(xb, mb, tpad, tl,
                                             list(sub.target_text), verbose=False)
        dt = time.time() - t0
        x_adv = int16_rt(torch.clamp(xb + best, -1, 1)) * mb
        snrs = snr_db_b(xb, x_adv, mb)
        hyp = decode_b(x_adv, mb)
        for i, (_, r) in enumerate(sub.iterrows()):
            ok = not np.isnan(eps[i])
            results.append(dict(
                utt_id=r["utt_id"], tag=cfg["tag"], rung=cfg["rung"],
                edit_type=EDIT_TYPE, checkpoint=cfg["checkpoint"],
                duration=r["duration"], source_text=r["text"],
                target_text=r["target_text"], neg_token=r["neg_token"],
                ctl_token=r["ctl_token"],
                source_tokens=int(r["source_tokens"]),
                target_tokens=int(r["target_tokens"]),
                eps50=float(eps[i]) if ok else None, censored=not ok,
                bracket_width=float(width[i]), snr_db=float(snrs[i]),
                decoded=(hyp[i] if ok else None),
                exact_match=(bool(hyp[i] == r["target_text"]) if ok else False),
                model_dir=MODEL_DIR))
        pd.DataFrame(results).to_csv(OUT, index=False)
        print(f"    batch {bi+1}/{len(batches)}  {dt/60:4.1f} min  "
              f"{int((~np.isnan(eps)).sum())}/{len(sub)} uncensored")
    df = pd.DataFrame(results)
    asr = float((~df.censored.astype(bool)).mean() * 100) if len(df) else None
    summary.append(dict(tag=cfg["tag"], rung=cfg["rung"], n=len(df), asr=asr))
    print(f"  {EDIT_TYPE} success {asr:.1f}%" if asr is not None else "  nothing measured")
    try:
        del model, processor
    except NameError:
        pass
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

print(f"\nSESSION DONE in {(time.time()-SESSION_START)/3600:.2f} h")

## 6 · T1 against T6 — the paired comparison that decides the section

In [ ]:
CEIL = EPS_HI

def load_eps(fn):
    """Read an eps50 CSV from any mount depth. Censored rows -> the ceiling."""
    for d in range(1, 5):
        for p in sorted(glob.glob("/kaggle/input/" + "*/" * d + fn)):
            df = pd.read_csv(p)
            df["eps_imputed"] = df.apply(
                lambda r: CEIL if str(r.censored).lower() in ("true", "1")
                else float(r.eps50), axis=1)
            return df.set_index("utt_id"), p
    for p in [f"{WORK}/{fn}"]:
        if os.path.exists(p):
            df = pd.read_csv(p)
            df["eps_imputed"] = df.apply(
                lambda r: CEIL if str(r.censored).lower() in ("true", "1")
                else float(r.eps50), axis=1)
            return df.set_index("utt_id"), p
    return None, None


print("Paired T1 vs T6 on the same utterances. Both are one-word deletions, so")
print("whatever is left is about the WORD, not about the size of the edit.\n")

for cfg in QUEUE:
    tag = cfg["tag"]
    t6, p6 = load_eps(f"eps50_rung{cfg['rung']}_{cfg['checkpoint']}"
                      f"_T6_n{len(ev)}.csv")
    # The named file first, then the other notebook's convention as a fallback,
    # so a rename upstream degrades to a clear message rather than a wrong pair.
    t1d, p1 = None, None
    for cand in (cfg["t1_csv"],
                 f"eps50_rung{cfg['rung']}_{cfg['checkpoint']}_T1_n38.csv",
                 f"eps50_rung{cfg['rung']}_{cfg['checkpoint']}_T1.csv"):
        t1d, p1 = load_eps(cand)
        if t1d is not None: break
    print("=" * 70)
    print(tag)
    if t6 is None:
        print("  T6 not measured yet."); continue
    if t1d is None:
        print(f"  T1 result not found: looked for '{cfg['t1_csv']}'.")
        print(f"  Attach EITHER the dataset '{cfg['t1_dataset']}', OR -- if that")
        print( "  dataset was never published -- the notebook output that wrote it")
        print( "  (Add Input -> Your Work -> Notebooks). Both mount at a depth this")
        print( "  reads. Without it there is no paired comparison, only a standalone")
        print(f"  T6 success rate of {100*(~t6.censored.astype(bool)).mean():.1f}%.")
        continue
    shared = sorted(set(t1d.index) & set(t6.index))
    print(f"  {len(shared)} shared utterances   (T1 from {os.path.basename(p1)})")
    a = t1d.loc[shared, "eps_imputed"].astype(float)
    b = t6.loc[shared, "eps_imputed"].astype(float)
    ratio = (b / a)
    cheaper = int((a < b).sum())
    print(f"  T1 success {100*(~t1d.loc[shared].censored.astype(bool)).mean():5.1f}%"
          f"   T6 success {100*(~t6.loc[shared].censored.astype(bool)).mean():5.1f}%")
    print(f"  T1 cheaper than T6 on {cheaper}/{len(shared)} utterances")
    print(f"  median per-utterance ratio T6/T1 = {float(ratio.median()):.2f}x")

    # Exact two-sided sign test: no distributional assumption, and it is the
    # test the direction claim actually needs.
    from math import comb
    n = int((a != b).sum()); k = min(cheaper, n - cheaper)
    p = min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0
    print(f"  sign test: n={n} discordant, p = {p:.4g}")
    print()
    if float(ratio.median()) > 2 and p < 0.05:
        print("  -> Negation deletion is substantially cheaper than a matched")
        print("     one-word deletion. That is a Bengali-specific finding and the")
        print("     section keeps its claim.")
    elif p >= 0.05:
        print("  -> T1 and T6 cost about the same. The 9.34x against T5 is an")
        print("     EDIT-DISTANCE result, not a negation result. Retitle the")
        print("     section and drop the linguistic reading. The security point")
        print("     is unaffected: reversing a sentence's meaning still costs a")
        print("     ninth of rewriting it.")
    else:
        print("  -> Direction is significant but the margin is small. Report the")
        print("     ratio and let the reader judge; do not lean on it.")

PROV.update(notebook=NOTEBOOK, edit_type=EDIT_TYPE, summary=summary,
            n_targets=int(len(ev)), eps_lo=EPS_LO, eps_hi=EPS_HI,
            bisect_rounds=BISECT_ROUNDS,
            session_hours=round((time.time()-SESSION_START)/3600, 2))
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), ensure_ascii=False,
          indent=2, default=str)
json.dump({"title": "negation-control-t6", "id": "YOURUSERNAME/negation-control-t6",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
for f in sorted(os.listdir(WORK)):
    print("   ", f)
print("\nCHECK THIS FILE LIST before publishing: publishing REPLACES the dataset.")